# *Attention Is All You Need*: el Transformer en PyTorch

Este cuaderno implementa, pieza por pieza, la arquitectura del Transformer original ([Vaswani et al., 2017](https://arxiv.org/abs/1706.03762)). Cada sección presenta un componente, su diagrama y su implementación.

<img src="img/transformer.png" width="380">

1. Embedding posicional
2. Atención multi-cabeza
3. Capa del encoder
4. Capa del decoder
5. Pila de $N$ capas: encoder y decoder
6. Transformer completo
7. Ejemplo de uso: de tokens a probabilidades

**Notación:** `B` = tamaño del batch, `L` = longitud de la secuencia, `d_model` = dimensión de los embeddings, `h` = número de cabezas.

In [1]:
from copy import deepcopy

import torch
import torch.nn as nn

torch.manual_seed(42)

/Users/danielvillarraga/Dropbox/2026-02/Deep Learning/Semana 13/.venv/lib/python3.14/site-packages/torch/_subclasses/functional_tensor.py:377: UserWarning: Failed to initialize NumPy: No module named 'numpy' (Triggered internally at /Users/runner/work/pytorch/pytorch/torch/csrc/utils/tensor_numpy.cpp:82.)
  cpu = _conversion_method_template(device=torch.device("cpu"))


## 1. Embedding posicional

La atención no tiene noción de orden: si se permutan los tokens, la salida se permuta igual. Para que el modelo sepa *dónde* está cada token, se suma un vector de posición a cada embedding.

<img src="img/embedding.png" width="450">

El artículo original usa senos y cosenos fijos. Aquí usamos una versión **entrenable**: una matriz $P \in \mathbb{R}^{L_{max} \times d_{model}}$ que se aprende junto con el resto del modelo, como en BERT o GPT-2.

$$Z = \text{Dropout}(X + P_{[:L]})$$

In [2]:
class PositionalEmbedding(nn.Module):
    def __init__(self, max_length, embed_dim, dropout=0.1):
        super().__init__()
        # Una fila aprendible por posición, inicializada con valores pequeños
        self.pos_embed = nn.Parameter(torch.randn(max_length, embed_dim) * 0.02)
        self.dropout = nn.Dropout(dropout)

    def forward(self, X):
        # X: (B, L, d_model). Solo se usan las primeras L posiciones
        return self.dropout(X + self.pos_embed[:X.size(1)])

In [3]:
pos_embed = PositionalEmbedding(max_length=100, embed_dim=512)
X = torch.randn(2, 10, 512)  # (B, L, d_model)
pos_embed(X).shape

torch.Size([2, 10, 512])

## 2. Atención multi-cabeza

### Atención de producto punto escalado

Cada token genera una *query* ($Q$), una *key* ($K$) y un *value* ($V$). La similitud entre queries y keys define cuánto atiende cada token a los demás:

$$\text{Attention}(Q, K, V) = \text{softmax}\left(\frac{QK^\top}{\sqrt{d_k}}\right)V$$

Se divide entre $\sqrt{d_k}$ para que los productos punto no crezcan con la dimensión y saturen el softmax.

<img src="img/scaled_dot_product.png" width="160">

### Múltiples cabezas

En lugar de una sola atención, se proyectan $Q$, $K$ y $V$ en $h$ subespacios de dimensión $d_k = d_{model}/h$. Cada cabeza aprende un tipo de relación distinto; al final se concatenan y se proyectan de vuelta:

$$\text{MultiHead}(Q, K, V) = \text{Concat}(\text{head}_1, \dots, \text{head}_h)\,W^O$$

<img src="img/multi_head.png" width="250">

### Máscaras

Las posiciones con `True` reciben $-\infty$ antes del softmax, así que su peso queda en cero:

- `attn_mask` `(Lq, Lk)`: bloquea pares de posiciones (p. ej., que el decoder vea el futuro).
- `key_padding_mask` `(B, Lk)`: ignora los tokens de *padding* de cada secuencia.

In [4]:
class MultiheadAttention(nn.Module):
    def __init__(self, embed_dim, num_heads, dropout=0.1):
        super().__init__()
        self.h = num_heads
        self.d = embed_dim // num_heads  # dimensión por cabeza (d_k)
        self.q_proj = nn.Linear(embed_dim, embed_dim)
        self.k_proj = nn.Linear(embed_dim, embed_dim)
        self.v_proj = nn.Linear(embed_dim, embed_dim)
        self.out_proj = nn.Linear(embed_dim, embed_dim)
        self.dropout = nn.Dropout(dropout)

    def split_heads(self, X):
        # (B, L, h × d) -> (B, h, L, d)
        return X.view(X.size(0), X.size(1), self.h, self.d).transpose(1, 2)

    def forward(self, query, key, value, attn_mask=None, key_padding_mask=None):
        q = self.split_heads(self.q_proj(query))  # (B, h, Lq, d)
        k = self.split_heads(self.k_proj(key))  # (B, h, Lk, d)
        v = self.split_heads(self.v_proj(value))  # (B, h, Lv, d) con Lv=Lk
        scores = q @ k.transpose(2, 3) / self.d**0.5  # (B, h, Lq, Lk)

        # Aplicación de máscaras:
        if attn_mask is not None:
            scores = scores.masked_fill(attn_mask, -torch.inf)  # (B, h, Lq, Lk)
        if key_padding_mask is not None:
            mask = key_padding_mask.unsqueeze(1).unsqueeze(2)  # (B, 1, 1, Lk)
            scores = scores.masked_fill(mask, -torch.inf)  # (B, h, Lq, Lk)

        weights = scores.softmax(dim=-1)  # (B, h, Lq, Lk)
        Z = self.dropout(weights) @ v  # (B, h, Lq, d)
        Z = Z.transpose(1, 2)  # (B, Lq, h, d)
        Z = Z.reshape(Z.size(0), Z.size(1), self.h * self.d)  # (B, Lq, h × d)
        return (self.out_proj(Z), weights)  # (B, Lq, h × d)

In [5]:
mha = MultiheadAttention(embed_dim=512, num_heads=8)
X = torch.randn(2, 10, 512)

# Self-attention: query, key y value son la misma secuencia
out, weights = mha(X, X, X)
print("Salida:", out.shape)  # (B, Lq, d_model)
print("Pesos: ", weights.shape)  # (B, h, Lq, Lk)
print("Cada fila de pesos suma 1:", torch.allclose(weights.sum(-1), torch.ones(1)))

Salida: torch.Size([2, 10, 512])
Pesos:  torch.Size([2, 8, 10, 10])
Cada fila de pesos suma 1: True


## 3. Capa del encoder

Cada capa del encoder tiene dos subcapas:

1. **Self-attention multi-cabeza:** cada token atiende a todos los tokens de la entrada.
2. **Red *feed-forward*:** dos capas lineales con ReLU, aplicadas a cada posición por separado.

$$\text{FFN}(x) = \max(0,\, xW_1 + b_1)\,W_2 + b_2$$

Ambas subcapas usan una **conexión residual** seguida de **LayerNorm** (*Add & Norm*):

$$Z = \text{LayerNorm}(X + \text{Dropout}(\text{Subcapa}(X)))$$

<img src="img/encoder.png" width="250">

In [9]:
class TransformerEncoderLayer(nn.Module):
    def __init__(self, d_model, nhead, dim_feedforward=2048, dropout=0.1):
        super().__init__()
        self.self_attn = MultiheadAttention(d_model, nhead, dropout)
        self.linear1 = nn.Linear(d_model, dim_feedforward)
        self.dropout = nn.Dropout(dropout)
        self.linear2 = nn.Linear(dim_feedforward, d_model)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)

    def forward(self, src, src_mask=None, src_key_padding_mask=None):
        # Subcapa 1: self-attention + Add & Norm
        attn, _ = self.self_attn(src, src, src, attn_mask=src_mask,
                                 key_padding_mask=src_key_padding_mask)
        Z = self.norm1(src + self.dropout(attn))
        # Subcapa 2: feed-forward + Add & Norm
        ff = self.dropout(self.linear2(self.dropout(self.linear1(Z).relu())))
        return self.norm2(Z + ff)

In [10]:
enc_layer = TransformerEncoderLayer(d_model=512, nhead=8)
src = torch.randn(2, 10, 512)
enc_layer(src).shape  # la forma se conserva: (B, Ls, d_model)

torch.Size([2, 10, 512])

## 4. Capa del decoder

El decoder tiene tres subcapas, cada una con *Add & Norm*:

1. **Self-attention enmascarada:** cada token solo puede atender a sí mismo y a los anteriores. Así el modelo no "ve el futuro" al predecir el siguiente token.
2. **Atención cruzada (*encoder-decoder*):** las queries vienen del decoder; las keys y values vienen de la salida del encoder (`memory`). Aquí el decoder consulta la oración de entrada.
3. **Red *feed-forward*:** igual que en el encoder.

<img src="img/decoder.png" width="230">

In [8]:
class TransformerDecoderLayer(nn.Module):
    def __init__(self, d_model, nhead, dim_feedforward=2048, dropout=0.1):
        super().__init__()
        self.self_attn = MultiheadAttention(d_model, nhead, dropout)
        self.multihead_attn = MultiheadAttention(d_model, nhead, dropout)
        self.dropout = nn.Dropout(dropout)
        self.linear1 = nn.Linear(d_model, dim_feedforward)
        self.linear2 = nn.Linear(dim_feedforward, d_model)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.norm3 = nn.LayerNorm(d_model)

    def forward(self, tgt, memory, tgt_mask=None, memory_mask=None,
                tgt_key_padding_mask=None, memory_key_padding_mask=None):
        # Subcapa 1: self-attention enmascarada + Add & Norm
        attn1, _ = self.self_attn(tgt, tgt, tgt,
                                  attn_mask=tgt_mask,
                                  key_padding_mask=tgt_key_padding_mask)
        Z = self.norm1(tgt + self.dropout(attn1))
        # Subcapa 2: atención cruzada (Q del decoder, K y V del encoder)
        attn2, _ = self.multihead_attn(Z, memory, memory, attn_mask=memory_mask,
                                       key_padding_mask=memory_key_padding_mask)
        Z = self.norm2(Z + self.dropout(attn2))
        # Subcapa 3: feed-forward + Add & Norm
        ff = self.dropout(self.linear2(self.dropout(self.linear1(Z).relu())))
        return self.norm3(Z + ff)

La **máscara causal** es una matriz triangular superior: `True` en las posiciones futuras que se deben bloquear.

In [6]:
def causal_mask(L):
    # True por encima de la diagonal = posiciones futuras bloqueadas
    return torch.triu(torch.ones(L, L, dtype=torch.bool), diagonal=1)

causal_mask(5)

tensor([[False,  True,  True,  True,  True],
        [False, False,  True,  True,  True],
        [False, False, False,  True,  True],
        [False, False, False, False,  True],
        [False, False, False, False, False]])

In [11]:
dec_layer = TransformerDecoderLayer(d_model=512, nhead=8)
memory = torch.randn(2, 10, 512)  # salida del encoder: (B, Ls, d_model)
tgt = torch.randn(2, 7, 512)  # entrada del decoder: (B, Lt, d_model)

dec_layer(tgt, memory, tgt_mask=causal_mask(7)).shape  # (B, Lt, d_model)

torch.Size([2, 7, 512])

## 5. Pila de $N$ capas: encoder y decoder

El encoder y el decoder repiten su capa $N$ veces (en el artículo, $N = 6$). Con `deepcopy` cada copia tiene **sus propios pesos**. La salida de una capa es la entrada de la siguiente. Opcionalmente se aplica una LayerNorm final.

<img src="img/transformer.png" width="300">

In [12]:
class TransformerEncoder(nn.Module):
    def __init__(self, encoder_layer, num_layers, norm=None):
        super().__init__()
        # N copias independientes de la misma capa
        self.layers = nn.ModuleList([deepcopy(encoder_layer)
                                     for _ in range(num_layers)])
        self.norm = norm

    def forward(self, src, mask=None, src_key_padding_mask=None):
        Z = src
        for layer in self.layers:
            Z = layer(Z, mask, src_key_padding_mask)
        if self.norm is not None:
            Z = self.norm(Z)
        return Z

In [13]:
class TransformerDecoder(nn.Module):
    def __init__(self, decoder_layer, num_layers, norm=None):
        super().__init__()
        # N copias independientes de la misma capa
        self.layers = nn.ModuleList([deepcopy(decoder_layer)
                                     for _ in range(num_layers)])
        self.norm = norm

    def forward(self, tgt, memory, tgt_mask=None, memory_mask=None,
                tgt_key_padding_mask=None, memory_key_padding_mask=None):
        Z = tgt
        for layer in self.layers:
            # Todas las capas consultan la misma memoria del encoder
            Z = layer(Z, memory, tgt_mask, memory_mask,
                      tgt_key_padding_mask, memory_key_padding_mask)
        if self.norm is not None:
            Z = self.norm(Z)
        return Z

In [14]:
encoder = TransformerEncoder(enc_layer, num_layers=6)
decoder = TransformerDecoder(dec_layer, num_layers=6)

memory = encoder(src)
print("Memoria del encoder:", memory.shape)
print("Salida del decoder: ", decoder(tgt, memory, tgt_mask=causal_mask(7)).shape)

Memoria del encoder: torch.Size([2, 10, 512])
Salida del decoder:  torch.Size([2, 7, 512])


## 6. Transformer completo

La clase `Transformer` une las piezas:

1. El encoder procesa la secuencia de entrada `src` y produce la memoria.
2. El decoder procesa la secuencia objetivo `tgt` y consulta la memoria en cada capa.

Los hiperparámetros por defecto son los del modelo *base* del artículo: $d_{model} = 512$, $h = 8$, $N = 6$, $d_{ff} = 2048$. Igual que `nn.Transformer`, esta clase recibe embeddings, no tokens. Los embeddings y la capa de salida se agregan en la sección 7.

<img src="img/transformer.png" width="300">

In [15]:
class Transformer(nn.Module):
    def __init__(self, d_model=512, nhead=8, num_encoder_layers=6,
                 num_decoder_layers=6, dim_feedforward=2048, dropout=0.1):
        super().__init__()
        encoder_layer = TransformerEncoderLayer(d_model, nhead, dim_feedforward,
                                                dropout)
        norm1 = nn.LayerNorm(d_model)
        self.encoder = TransformerEncoder(encoder_layer, num_encoder_layers,
                                          norm1)
        decoder_layer = TransformerDecoderLayer(d_model, nhead, dim_feedforward,
                                                dropout)
        norm2 = nn.LayerNorm(d_model)
        self.decoder = TransformerDecoder(decoder_layer, num_decoder_layers,
                                          norm2)

    def forward(self, src, tgt, src_mask=None, tgt_mask=None, memory_mask=None,
                src_key_padding_mask=None, tgt_key_padding_mask=None,
                memory_key_padding_mask=None):
        memory = self.encoder(src, src_mask, src_key_padding_mask)
        output = self.decoder(tgt, memory, tgt_mask, memory_mask,
                              tgt_key_padding_mask, memory_key_padding_mask)
        return output

In [16]:
model = Transformer()

def count_params(m):
    return sum(p.numel() for p in m.parameters())

# Mismo número de parámetros que la implementación oficial de PyTorch
print(f"Nuestro Transformer: {count_params(model):,}")
print(f"nn.Transformer:      {count_params(nn.Transformer(batch_first=True)):,}")

model(src, tgt, tgt_mask=causal_mask(7)).shape

Nuestro Transformer: 44,140,544
nn.Transformer:      44,140,544


torch.Size([2, 7, 512])

## 7. Ejemplo de uso: de tokens a probabilidades

Para pasar de tokens a predicciones faltan las piezas de los extremos del diagrama:

- **Entrada:** `nn.Embedding` convierte IDs en vectores y `PositionalEmbedding` agrega la posición.
- **Salida:** una capa `Linear` proyecta a `vocab_size` y el softmax da la probabilidad del siguiente token.

Durante el entrenamiento, el decoder recibe la oración objetivo **desplazada a la derecha** (*shifted right*): empieza con `<sos>` y debe predecir el token siguiente en cada posición. Además, las secuencias del batch se rellenan con *padding* (ID `0`), que se ignora con `key_padding_mask`.

In [17]:
vocab_size, d_model, max_length, PAD = 1000, 128, 50, 0

embed = nn.Embedding(vocab_size, d_model, padding_idx=PAD)
pos = PositionalEmbedding(max_length, d_model)
transformer = Transformer(d_model=d_model, nhead=4, num_encoder_layers=2,
                          num_decoder_layers=2, dim_feedforward=256)
output_proj = nn.Linear(d_model, vocab_size)

# Batch de 2 oraciones con padding (ID 0) al final
src_ids = torch.tensor([[5, 23, 87, 4, 0, 0],
                        [9, 41, 12, 33, 7, 2]])  # (B, Ls)
tgt_ids = torch.tensor([[1, 64, 18, 0],
                        [1, 77, 25, 90]])  # (B, Lt), 1 = <sos>

# Máscaras: True = posición ignorada
src_pad = src_ids == PAD
tgt_pad = tgt_ids == PAD

out = transformer(pos(embed(src_ids)), pos(embed(tgt_ids)),
                  tgt_mask=causal_mask(tgt_ids.size(1)),
                  src_key_padding_mask=src_pad,
                  tgt_key_padding_mask=tgt_pad,
                  memory_key_padding_mask=src_pad)  # (B, Lt, d_model)

logits = output_proj(out)  # (B, Lt, vocab_size)
probs = logits.softmax(dim=-1)
print("Logits:", logits.shape)
print("Token más probable en cada posición:\n", probs.argmax(dim=-1))

Logits: torch.Size([2, 4, 1000])
Token más probable en cada posición:
 tensor([[873, 649, 919, 475],
        [873, 181, 832, 505]])


Sin entrenar, las predicciones son aleatorias. Para entrenar el modelo se compara `logits` con la oración objetivo desplazada a la izquierda usando `nn.CrossEntropyLoss(ignore_index=PAD)`.